# Article 1 — Phase 1 : statistiques, splits gelés, sonde temporelle, baselines

**Papier :** *A Leakage-Audited Benchmark of Deep and Ensemble Detectors on the GeNIS 2025 Corpus*.

**Ce que fait ce notebook (E1 partiel, E2 partiel, sonde E3) :**
1. Statistiques des 4 intervalles (5/10/30/60 s) — table du papier §3.
2. Tranche 60 s : fusion des 3 variantes bénignes → **9 classes**, deux jeux de features :
   - `full`  = features numériques **avec** colonnes positionnelles (StartTime, LastTime, Rank, Seq) → condition « avant audit » ;
   - `clean` = sans positionnelles ni quasi-identifiants → condition « après audit (candidate) ».
3. **Splits gelés** : stratifié 60/20/20 × 5 graines + split **chronologique** (les indices sont exportés, plus jamais régénérés).
4. **Sonde temporelle** : classifieur sur StartTime seul (résultat attendu ≈ 99 % — c'est le cœur de RQ2).
5. Baselines : classe majoritaire, régression logistique, Random Forest, XGBoost, LightGBM (multiclasse ; les métriques binaires sont dérivées de p(benign)).
6. Trio profond **RNN / CNN / DNN — architectures STRICTEMENT identiques au notebook BAg-IDS** (provenance des détecteurs du papier système).

**Consignes d'exécution :**
- Runtime **GPU** conseillé (le trio profond en profite ; le reste est CPU).
- `2-flows.zip` doit être dans `MyDrive/GeNIS/` (sinon la cellule 3 le télécharge depuis Zenodo).
- Exécution : *Exécution → Tout exécuter*. Durée ≈ 2–4 h. Les résultats sont sauvegardés sur Drive **après chaque bloc** : une déconnexion ne perd que le bloc en cours.

**À me rapporter à la fin :** le fichier `article1_phase1_results.json` (téléchargé automatiquement) + toute sortie anormale.


In [ ]:
# 1) Environnement
import sys, platform, time, json, hashlib, os, glob, pathlib, shutil
import numpy as np, pandas as pd, sklearn
import tensorflow as tf
try:
    import xgboost, lightgbm
except ImportError:
    %pip -q install xgboost lightgbm
    import xgboost, lightgbm

print("python     :", platform.python_version())
print("numpy      :", np.__version__, "| pandas :", pd.__version__)
print("sklearn    :", sklearn.__version__)
print("xgboost    :", xgboost.__version__, "| lightgbm :", lightgbm.__version__)
print("tensorflow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU') or "AUCUN (ok pour les arbres, lent pour le trio profond)")

RESULTS = {"phase": 1, "created": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "env": {"python": platform.python_version(), "numpy": np.__version__,
                   "pandas": pd.__version__, "sklearn": sklearn.__version__,
                   "xgboost": xgboost.__version__, "lightgbm": lightgbm.__version__,
                   "tensorflow": tf.__version__,
                   "gpu": bool(tf.config.list_physical_devices('GPU'))}}


In [ ]:
# 2) Données : Drive d'abord, Zenodo en secours
from google.colab import drive
drive.mount('/content/drive')

WORK = pathlib.Path("/content/genis"); WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
SAVE_DIR = pathlib.Path("/content/drive/MyDrive/GeNIS/article1")  # sorties persistantes
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def save_results():
    (SAVE_DIR / "article1_phase1_results.json").write_text(
        json.dumps(RESULTS, indent=1, default=float), encoding="utf-8")
    print(f"[checkpoint] resultats sauvegardes -> {SAVE_DIR/'article1_phase1_results.json'}")

drive_zip = "/content/drive/MyDrive/GeNIS/2-flows.zip"
if not pathlib.Path("flows").exists():
    if pathlib.Path(drive_zip).exists():
        print("copie depuis Drive…"); shutil.copy(drive_zip, "2-flows.zip")
    else:
        print("telechargement Zenodo (~380 Mo)…")
        !wget -q --show-progress "https://zenodo.org/records/14919237/files/2-flows.zip?download=1" -O 2-flows.zip
    !unzip -o -q 2-flows.zip -d flows

csvs = sorted(glob.glob("flows/**/*.csv", recursive=True))
print(f"{len(csvs)} CSV trouves")
assert csvs, "Aucun CSV : verifier le zip"


In [ ]:
# 3) Statistiques des 4 intervalles (classe = nom de fichier ; comptage de lignes, rapide)
INTERVALS = ["5", "10", "30", "60"]
interval_stats = {}
for iv in INTERVALS:
    files = [c for c in csvs if f"flows-{iv}-sec" in c]
    counts = {}
    for f in files:
        cls = pathlib.Path(f).stem.replace("attack-", "").replace("benign-", "benign|")
        n = sum(1 for _ in open(f, "rb")) - 1
        counts[pathlib.Path(f).stem] = n
    total = sum(counts.values())
    benign = sum(v for k, v in counts.items() if k.startswith("benign"))
    interval_stats[iv] = {"files": counts, "total_flows": total,
                          "benign_flows": benign, "benign_share": benign / total}
    print(f"intervalle {iv:>2s} s : {total:>9,} flux | benin {benign:>7,} ({benign/total:.1%})")

RESULTS["interval_stats"] = interval_stats
save_results()
# --> Table "per-interval statistics" du papier (la part de benin VARIE avec l'intervalle)


In [ ]:
# 4) Tranche 60 s : chargement, labels 9 classes, metadonnees temporelles
SIXTY = [c for c in csvs if "flows-60-sec" in c]
df = pd.concat([pd.read_csv(c, low_memory=False) for c in SIXTY], ignore_index=True)
print("brut :", df.shape)

LABEL_COL = "SubCategoryLabel"
assert LABEL_COL in df.columns, f"{LABEL_COL} absent — colonnes : {list(df.columns)[:40]}"
y_sub = df[LABEL_COL].astype(str).str.strip()

# Fusion des 3 variantes benignes -> 9 classes (protocole gele, coherent avec BAg-IDS §6.9)
y9_raw = y_sub.where(~y_sub.str.startswith("benign"), "benign")
print("\nDistribution naturelle (9 classes) :")
print(y9_raw.value_counts())

# Metadonnees temporelles : necessaires au split chronologique et a la sonde,
# JAMAIS features en condition 'clean'.
assert "StartTime" in df.columns, "StartTime absent — adapter"
t_start = pd.to_numeric(df["StartTime"], errors="coerce")
assert t_start.notna().all(), "StartTime non numerique — adapter"
RESULTS["slice60"] = {"n_flows": int(len(df)),
                      "class_counts_9": y9_raw.value_counts().to_dict(),
                      "benign_share": float((y9_raw == "benign").mean()),
                      "capture_span_hours": float((t_start.max() - t_start.min()) / 3600)}
print(f"\npart de benin (9 classes, test attendu ~identique) : {(y9_raw=='benign').mean():.1%}")
print(f"duree de capture couverte : {(t_start.max()-t_start.min())/3600:.1f} h")


In [ ]:
# 5) Jeux de features EXPLICITES (plus de patterns : le bug 'Idl'~'id' du
#    notebook BAg-IDS ecartait par accident SIntPktIdl/DIntPktIdl, et laissait
#    passer StartTime/LastTime — corrige ici, liste par liste).

# (a) identifiants / quasi-identifiants : jamais features (aucune condition)
IDENTIFIERS = [c for c in [
    "FlowID", "AutoId", "SrcAddr", "DstAddr", "Ssaddr", "Sdaddr",
    "SrcMac", "DstMac", "SrcOui", "DstOui", "Sport", "Dport",
    "sIpId", "dIpId", "sMpls", "dMpls", "sAS", "dAS", "iAS",
    "sCo", "dCo", "sVid", "dVid",
] if c in df.columns]

# (b) positionnelles : EXCLUES en 'clean', INCLUSES en 'full' (condition avant-audit)
POSITIONAL = [c for c in ["StartTime", "LastTime", "Rank", "Seq"] if c in df.columns]

# (c) labels
LABELS = [c for c in ["BinaryLabel", "CategoryLabel", "SubCategoryLabel"] if c in df.columns]

num = df.drop(columns=IDENTIFIERS + LABELS, errors="ignore").select_dtypes(include=[np.number])
num = num.replace([np.inf, -np.inf], np.nan)
const = num.nunique(dropna=True); const_cols = const[const <= 1].index.tolist()
num = num.drop(columns=const_cols).fillna(0.0).astype(np.float32)

FEATURES_FULL  = list(num.columns)                                  # avec positionnelles
FEATURES_CLEAN = [c for c in FEATURES_FULL if c not in POSITIONAL]  # sans

print(f"identifiants exclus partout ({len(IDENTIFIERS)}) : {IDENTIFIERS}")
print(f"positionnelles ({len(POSITIONAL)}) : {POSITIONAL}")
print(f"constantes/vides ecartees ({len(const_cols)}) : {const_cols}")
print(f"\nfeatures FULL  : {len(FEATURES_FULL)}")
print(f"features CLEAN : {len(FEATURES_CLEAN)}")
for c in ["SIntPktIdl", "SIntIdlDist", "DIntPktIdl", "DIntIdlDist", "IdleTime", "RunTime"]:
    print(f"  {c:14s} present en clean : {c in FEATURES_CLEAN}")

RESULTS["features"] = {"identifiers_excluded": IDENTIFIERS, "positional": POSITIONAL,
                       "constant_dropped": const_cols,
                       "full": FEATURES_FULL, "clean": FEATURES_CLEAN}

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder(); y = le.fit_transform(y9_raw)
CLASS_NAMES = list(le.classes_); BENIGN_IDX = CLASS_NAMES.index("benign")
X_all = num  # DataFrame float32 ; on selectionne les colonnes par condition
print("\nclasses :", CLASS_NAMES)


In [ ]:
# 6) SPLITS GELES : 5 stratifies + 1 chronologique (exportes, plus jamais regeneres)
from sklearn.model_selection import train_test_split

SEEDS = [1, 2, 3, 4, 5]
idx_all = np.arange(len(y))
splits = {}
for s in SEEDS:
    itr, itmp = train_test_split(idx_all, test_size=0.40, random_state=s, stratify=y)
    iva, ite  = train_test_split(itmp,   test_size=0.50, random_state=s, stratify=y[itmp])
    splits[f"strat_seed{s}"] = (itr, iva, ite)

order = np.argsort(t_start.values, kind="stable")
n = len(order); a, b = int(0.60 * n), int(0.80 * n)
splits["chrono"] = (order[:a], order[a:b], order[b:])

def split_hash(tr, va, te):
    h = hashlib.sha256()
    for arr in (tr, va, te): h.update(np.ascontiguousarray(np.sort(arr)).tobytes())
    return h.hexdigest()[:16]

np.savez_compressed(SAVE_DIR / "frozen_splits_60s.npz",
                    **{f"{k}_{p}": v for k, (tr, va, te) in splits.items()
                       for p, v in zip(("train", "val", "test"), (tr, va, te))})
RESULTS["splits"] = {k: {"hash": split_hash(*v),
                         "sizes": [len(v[0]), len(v[1]), len(v[2])]} for k, v in splits.items()}
print("splits geles ->", SAVE_DIR / "frozen_splits_60s.npz")

# Presence des classes dans le split chronologique : LA table qui montre que les
# fenetres d'attaque sont disjointes dans le temps (resultat, pas bug).
tab = pd.DataFrame({p: pd.Series(y[ix]).value_counts().reindex(range(len(CLASS_NAMES)), fill_value=0).values
                    for p, ix in zip(("train", "val", "test"), splits["chrono"])},
                   index=CLASS_NAMES)
print("\nRepartition des classes dans le split CHRONOLOGIQUE :"); print(tab)
RESULTS["chrono_class_table"] = tab.to_dict()
save_results()


In [ ]:
# 7) SONDE TEMPORELLE (coeur de RQ2) : StartTime seul, puis positionnelles seules
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def probe(cols, split_key):
    tr, va, te = splits[split_key]
    Xp = df[cols].astype(np.float64).values
    clf = DecisionTreeClassifier(random_state=1).fit(Xp[tr], y[tr])
    return accuracy_score(y[te], clf.predict(Xp[te]))

chance = pd.Series(y).value_counts(normalize=True).max()
probes = {"chance_majority": float(chance)}
for name, cols in [("starttime_only", ["StartTime"]), ("positional_all", POSITIONAL)]:
    accs = [probe(cols, f"strat_seed{s}") for s in SEEDS]
    probes[name] = {"stratified_mean": float(np.mean(accs)),
                    "stratified_std": float(np.std(accs)),
                    "chrono": float(probe(cols, "chrono"))}
    print(f"{name:15s} | stratifie {np.mean(accs):.4f} +/- {np.std(accs):.4f} "
          f"| chrono {probes[name]['chrono']:.4f} | hasard {chance:.4f}")

RESULTS["shortcut_probes"] = probes
save_results()
# Attendu : ~0.99 en stratifie (le raccourci), effondrement en chrono. Ce contraste
# est la premiere figure du papier.


In [ ]:
# 8) Metriques communes (protocole : metriques binaires derivees de p(benign))
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef,
                             average_precision_score)
from sklearn.preprocessing import RobustScaler

def evaluate(y_true, probs, fit_time, pred_time):
    pred = probs.argmax(1)
    is_att = (y_true != BENIGN_IDX); pred_att = (pred != BENIGN_IDX)
    p_att = 1.0 - probs[:, BENIGN_IDX]
    fpr = float(pred_att[~is_att].mean()) if (~is_att).any() else None
    fnr = float((~pred_att[is_att]).mean()) if is_att.any() else None
    return {"accuracy": float(accuracy_score(y_true, pred)),
            "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
            "mcc": float(matthews_corrcoef(y_true, pred)),
            "per_class_f1": {CLASS_NAMES[i]: float(v) for i, v in enumerate(
                f1_score(y_true, pred, average=None, zero_division=0,
                         labels=range(len(CLASS_NAMES))))},
            "binary": {"detection_f1": float(f1_score(is_att, pred_att, zero_division=0)),
                       "fpr": fpr, "fnr": fnr,
                       "pr_auc": float(average_precision_score(is_att, p_att)) if is_att.any() else None},
            "fit_time_s": round(fit_time, 2), "predict_time_s": round(pred_time, 3),
            "throughput_flows_per_s": round(len(y_true) / max(pred_time, 1e-9))}

def make_xy(cols, split_key):
    tr, va, te = splits[split_key]
    X = X_all[cols].values
    sc = RobustScaler().fit(X[tr])                     # scaler sur TRAIN uniquement
    out = [np.nan_to_num(sc.transform(X[ix]), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
           for ix in (tr, va, te)]
    return out, (y[tr], y[va], y[te])
print("helpers OK")


In [ ]:
# 9) Baselines + arbres : 5 modeles x 2 conditions x (5 stratifies + chrono)
#    (multiclasse 9 ; hyperparametres par defaut declares — la recherche a budget
#     egal viendra en phase 2 pour le tableau final)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

def zoo():
    return {
        "majority": DummyClassifier(strategy="most_frequent"),
        "logreg":   LogisticRegression(max_iter=1000, n_jobs=-1),
        "rf":       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
        "xgboost":  XGBClassifier(tree_method="hist", n_estimators=300, max_depth=8,
                                  learning_rate=0.1, n_jobs=-1, random_state=0,
                                  eval_metric="mlogloss"),
        "lightgbm": LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                                   n_jobs=-1, random_state=0, verbose=-1),
    }

PROBS_DIR = pathlib.Path("/content/probs"); PROBS_DIR.mkdir(exist_ok=True)
RESULTS.setdefault("models", {})
CONDITIONS = {"full": FEATURES_FULL, "clean": FEATURES_CLEAN}
SPLIT_KEYS = [f"strat_seed{s}" for s in SEEDS] + ["chrono"]

for cond, cols in CONDITIONS.items():
    for sk in SPLIT_KEYS:
        (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(cols, sk)
        for mname, model in zoo().items():
            key = f"{mname}|{cond}|{sk}"
            if key in RESULTS["models"]:
                continue  # reprise apres deconnexion
            t0 = time.time(); model.fit(Xtr, ytr); fit_t = time.time() - t0
            t0 = time.time(); pte = model.predict_proba(Xte); pred_t = time.time() - t0
            pva = model.predict_proba(Xva)
            RESULTS["models"][key] = evaluate(yte, pte, fit_t, pred_t)
            np.savez_compressed(PROBS_DIR / f"{mname}_{cond}_{sk}.npz",
                                probs_val=pva.astype(np.float16),
                                probs_test=pte.astype(np.float16))
            r = RESULTS["models"][key]
            fpr = r['binary']['fpr']; fpr_s = f"{fpr:.4%}" if fpr is not None else "n/a"
            print(f"{key:35s} acc {r['accuracy']:.4f} | mF1 {r['macro_f1']:.4f} "
                  f"| MCC {r['mcc']:.4f} | FPR {fpr_s}")
        save_results()


In [ ]:
# 10) Trio profond RNN / CNN / DNN — architectures IDENTIQUES au notebook
#     BAg-IDS (provenance des detecteurs du papier systeme ; ne pas modifier).
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight

C = len(CLASS_NAMES)

def build_dnn(F):
    return models.Sequential([layers.Input((F,)),
        layers.Dense(128, activation="relu"), layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),  layers.Dropout(0.2),
        layers.Dense(C, activation="softmax")], name="dnn")

def build_cnn(F):
    return models.Sequential([layers.Input((F, 1)),
        layers.Conv1D(64, 3, activation="relu", padding="same"), layers.MaxPooling1D(2),
        layers.Conv1D(32, 3, activation="relu", padding="same"), layers.Flatten(),
        layers.Dense(64, activation="relu"), layers.Dropout(0.3),
        layers.Dense(C, activation="softmax")], name="cnn")

def build_rnn(F):
    return models.Sequential([layers.Input((F, 1)),
        layers.SimpleRNN(64, activation="relu"), layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dense(C, activation="softmax")], name="rnn")

BUILDERS = {"rnn": build_rnn, "cnn": build_cnn, "dnn": build_dnn}
def shape_for(name, A): return A if name == "dnn" else A.reshape(-1, A.shape[1], 1)

# Budget phase 1 : clean x 5 graines (tableau E2) + full x graine 1 (avant/apres).
# Reduire N_SEEDS_DEEP si le temps manque — me le signaler dans le rapport.
N_SEEDS_DEEP = 5
DEEP_RUNS = [("clean", f"strat_seed{s}") for s in SEEDS[:N_SEEDS_DEEP]] + [("full", "strat_seed1")]

for cond, sk in DEEP_RUNS:
    cols = CONDITIONS[cond]
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(cols, sk)
    F = Xtr.shape[1]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(C), y=ytr)))
    for mname, build in BUILDERS.items():
        key = f"{mname}|{cond}|{sk}"
        if key in RESULTS["models"]:
            continue
        tf.keras.utils.set_random_seed(int(sk[-1]) if sk[-1].isdigit() else 1)
        m = build(F)
        m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        t0 = time.time()
        m.fit(shape_for(mname, Xtr), ytr, validation_data=(shape_for(mname, Xva), yva),
              epochs=30, batch_size=256, class_weight=cw, verbose=0,
              callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                 restore_best_weights=True)])
        fit_t = time.time() - t0
        t0 = time.time(); pte = m.predict(shape_for(mname, Xte), batch_size=1024, verbose=0)
        pred_t = time.time() - t0
        pva = m.predict(shape_for(mname, Xva), batch_size=1024, verbose=0)
        RESULTS["models"][key] = evaluate(yte, pte, fit_t, pred_t)
        np.savez_compressed(PROBS_DIR / f"{mname}_{cond}_{sk}.npz",
                            probs_val=pva.astype(np.float16), probs_test=pte.astype(np.float16))
        if cond == "clean" and sk == "strat_seed1":
            m.save(PROBS_DIR / f"{mname}_clean_seed1.keras")   # checkpoints de provenance
        r = RESULTS["models"][key]
        fpr = r['binary']['fpr']; fpr_s = f"{fpr:.4%}" if fpr is not None else "n/a"
        print(f"{key:35s} acc {r['accuracy']:.4f} | mF1 {r['macro_f1']:.4f} "
              f"| MCC {r['mcc']:.4f} | FPR {fpr_s} | {fit_t:.0f}s")
    save_results()


In [ ]:
# 11) Importance par permutation (LightGBM, condition FULL) : premiere preuve
#     de la liste noire — quelles colonnes portent le raccourci ?
from sklearn.inspection import permutation_importance
from lightgbm import LGBMClassifier

(Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(FEATURES_FULL, "strat_seed1")
lgbm = LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                      n_jobs=-1, random_state=0, verbose=-1).fit(Xtr, ytr)
rng = np.random.RandomState(0); sub = rng.choice(len(yte), size=min(20000, len(yte)), replace=False)
imp = permutation_importance(lgbm, Xte[sub], yte[sub], n_repeats=5, random_state=0,
                             n_jobs=-1, scoring="accuracy")
order_imp = np.argsort(-imp.importances_mean)
top = [(FEATURES_FULL[i], float(imp.importances_mean[i]), float(imp.importances_std[i]))
       for i in order_imp[:25]]
print(f"{'feature':22s} {'delta acc':>10s}")
for f, m_, s_ in top:
    tag = " <-- POSITIONNELLE" if f in POSITIONAL else ""
    print(f"{f:22s} {m_:>10.4f} +/- {s_:.4f}{tag}")
RESULTS["permutation_importance_lgbm_full"] = top
save_results()


In [ ]:
# 12) Archive des probabilites + telechargement du rapport
shutil.make_archive(str(SAVE_DIR / "article1_phase1_probs"), "zip", PROBS_DIR)
print("probs   ->", SAVE_DIR / "article1_phase1_probs.zip",
      f"({os.path.getsize(SAVE_DIR/'article1_phase1_probs.zip')/1e6:.0f} Mo)")
save_results()

from google.colab import files
files.download(str(SAVE_DIR / "article1_phase1_results.json"))
print("Termine. Envoyer article1_phase1_results.json (les .npz restent sur Drive pour la phase 2).")


## À me rapporter

1. **`article1_phase1_results.json`** (téléchargé à la fin — ou depuis `MyDrive/GeNIS/article1/`).
2. La **table chronologique** (cellule 6) : quelles classes sont absentes du test chrono.
3. Toute cellule en erreur, avec son message.

**Points de vigilance (me les signaler s'ils se produisent) :**
- Sonde StartTime < 95 % en stratifié → le raccourci est plus faible qu'attendu, on ajuste la narration ;
- un modèle `clean` > 99,9 % macro-F1 → chercher une colonne identifiante restante (on l'ajoutera à la liste noire) ;
- trio profond très en-dessous des arbres → normal sur tabulaire, c'est un résultat, pas un bug.

**Phase 2 (notebook suivant, après lecture de ces résultats) :** FT-Transformer, autoencodeur (leave-one-family-out), trio profond sur split chrono, exclusions guidées par l'importance de permutation → liste noire définitive, puis E4 multi-intervalles et E6 banc de coût.
